In [72]:
#import thư viện

import json
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier # Sử dụng Gradient Boosting
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, RocCurveDisplay, ConfusionMatrixDisplay

In [8]:
# Đọc file lịch sử đấu
df = pd.read_csv('matches.csv')
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 20397 entries, 0 to 20396
Data columns (total 26 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   match_id       20397 non-null  str  
 1   game_version   20397 non-null  str  
 2   game_duration  20397 non-null  int64
 3   blue_win       20397 non-null  int64
 4   blue_top_id    20397 non-null  int64
 5   blue_top_name  20397 non-null  str  
 6   blue_jg_id     20397 non-null  int64
 7   blue_jg_name   20397 non-null  str  
 8   blue_mid_id    20397 non-null  int64
 9   blue_mid_name  20397 non-null  str  
 10  blue_bot_id    20397 non-null  int64
 11  blue_bot_name  20397 non-null  str  
 12  blue_sup_id    20397 non-null  int64
 13  blue_sup_name  20397 non-null  str  
 14  red_top_id     20397 non-null  int64
 15  red_top_name   20397 non-null  str  
 16  red_jg_id      20397 non-null  int64
 17  red_jg_name    20397 non-null  str  
 18  red_mid_id     20397 non-null  int64
 19  red_mid_name   

,match_id,game_version,game_duration,blue_win,blue_top_id,blue_top_name,blue_jg_id,blue_jg_name,blue_mid_id,blue_mid_name,...,red_jg_id,red_jg_name,red_mid_id,red_mid_name,red_bot_id,red_bot_name,red_sup_id,red_sup_name,blue_champs,red_champs
0,KR_8453665812,16.22.834.8910,1839,0,240,Kled,163,Taliyah,84,Akali,...,64,LeeSin,134,Syndra,81,Ezreal,89,Leona,240;163;84;221;497,114;64;134;81;89
1,KR_8453642639,16.22.834.8910,914,1,2,Olaf,107,Rengar,10,Kayle,...,78,Poppy,517,Sylas,51,Caitlyn,99,Lux,2;107;10;202;111,24;78;517;51;99
2,KR_8453601923,16.22.834.8910,1647,0,2,Olaf,121,Khazix,10,Kayle,...,107,Rengar,3,Galio,236,Lucian,902,Milio,2;121;10;21;111,164;107;3;236;902
3,KR_8451415459,16.22.834.8910,1397,1,2,Olaf,5,XinZhao,245,Ekko,...,107,Rengar,103,Ahri,145,Kaisa,111,Nautilus,2;5;245;804;117,80;107;103;145;111
4,KR_8451375104,16.22.834.8910,1440,0,24,Jax,246,Qiyana,127,Lissandra,...,107,Rengar,103,Ahri,81,Ezreal,57,Maokai,24;246;127;145;40,80;107;103;81;57


In [12]:
# Đọc file metadata tướng
with open('champions.json', 'r') as champ:
    champ_info = {int(k): v for k, v in json.load(champ).items()}
champ_info

{266: {'id': 266,
  'name': 'Aatrox',
  'key': 'Aatrox',
  'tags': ['Fighter'],
  'info': {'attack': 8, 'defense': 4, 'magic': 3, 'difficulty': 4}},
 103: {'id': 103,
  'name': 'Ahri',
  'key': 'Ahri',
  'tags': ['Mage', 'Assassin'],
  'info': {'attack': 3, 'defense': 4, 'magic': 8, 'difficulty': 5}},
 84: {'id': 84,
  'name': 'Akali',
  'key': 'Akali',
  'tags': ['Assassin'],
  'info': {'attack': 5, 'defense': 3, 'magic': 8, 'difficulty': 7}},
 166: {'id': 166,
  'name': 'Akshan',
  'key': 'Akshan',
  'tags': ['Marksman', 'Assassin'],
  'info': {'attack': 0, 'defense': 0, 'magic': 0, 'difficulty': 0}},
 12: {'id': 12,
  'name': 'Alistar',
  'key': 'Alistar',
  'tags': ['Tank', 'Support'],
  'info': {'attack': 6, 'defense': 9, 'magic': 5, 'difficulty': 7}},
 799: {'id': 799,
  'name': 'Ambessa',
  'key': 'Ambessa',
  'tags': ['Fighter', 'Assassin'],
  'info': {'attack': 9, 'defense': 2, 'magic': 0, 'difficulty': 10}},
 32: {'id': 32,
  'name': 'Amumu',
  'key': 'Amumu',
  'tags': ['Tan

In [23]:
# Do id tướng không theo thứ tự -> tạo index từ 0 đến n-1
sorted_id = sorted(list(champ_info.keys()))
riotid_to_idx = {riotid: idx for idx, riotid in enumerate(sorted_id)}
idx_to_riotid = {idx:riotid for riotid, idx in riotid_to_idx.items()}
champ_count = len(sorted_id)

In [56]:
blue_cols = ['blue_top_id', 'blue_jg_id', 'blue_mid_id', 'blue_bot_id', 'blue_sup_id']
red_cols = ['red_top_id', 'red_jg_id', 'red_mid_id', 'red_bot_id', 'red_sup_id']
X = np.zeros((len(df), champ_count), dtype=np.int32)
y = df['blue_win'].values.astype(int)

# Gán nhãn 1 cho đội xanh, -1 cho đội đỏ
for rowid, (_, row) in enumerate(df.iterrows()):
    for c in blue_cols:
        X[rowid, riotid_to_idx[row[c]]] = 1
    for c in red_cols:
        X[rowid, riotid_to_idx[row[c]]] = -1

# Chia train, test
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify=y
)

# Nhân đôi dữ liệu: A xanh B đỏ: xanh thắng -> A đỏ B xanh: đỏ thắng
# => thêm một nửa tập dữ liệu X <=> -X, y <=> 1-y
X_train = np.vstack([X_train_raw, -X_train_raw])
y_train = np.concatenate([y_train_raw, 1-y_train_raw])

X_test = np.vstack([X_test_raw, -X_test_raw])
y_test = np.concatenate([y_test_raw, 1-y_test_raw])

array([0, 1, 1, ..., 0, 0, 0], shape=(8160,))

In [68]:
# Train model
model = HistGradientBoostingClassifier(
    max_iter = 250,
    learning_rate = 0.02,
    max_depth = 6,
    min_samples_leaf = 25,
    l2_regularization = 1.5,
    early_stopping = True,
    n_iter_no_change = 10,
    random_state = 42
)

model.fit(X_train, y_train)

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.02
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",250
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",6
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",25
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.5
,"early_stopping early_stopping: 'auto' or bool, default='auto'If 'auto', early stopping is enabled if the sample size is larger than10000 or if `X_val` and `y_val` are passed to `fit`. If True, early stoppingis enabled, otherwise early stopping is disabled... versionadded:: 0.23",True
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [75]:
# Đánh giá
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)
loss = log_loss(y_test, y_pred_proba)

print(acc * 100)
print(auc)
print(loss)

plt.show()

52.352941176470594
0.5333487420703575
0.6915919489915515


In [108]:
# Dự đoán team 5 vs 5
def pred(blue_team, red_team):

    if len(blue_team) != 5 or len(red_team) != 5:
        print('Mỗi đội đúng 5 tướng!')
        return
    
    proc = lambda s: s.lower().replace(" ", "").replace("'", "").replace(".", "")

    lookup = {}
    for riotid, info in champ_info.items():
        lookup[proc(info['name'])] = riotid
        lookup[proc(info['key'])] = riotid

    stat = np.zeros((1, champ_count), dtype=np.int32)

    try:
        for champ_name in blue_team:
            riotid = lookup[proc(champ_name)]
            stat[0][riotid_to_idx[riotid]] = 1
        for champ_name in red_team:
            riotid = lookup[proc(champ_name)]
            stat[0][riotid_to_idx[riotid]] = -1
    except KeyError as e:
        print('Không thấy tướng tên:', e)
        return

    prob_blue = model.predict_proba(stat)[0, 1]
    print("\n" + "=" * 65)
    prob_red = 1 - prob_blue
    print('⚔️ KẾT QUẢ DỰ ĐOÁN TỈ LỆ THẮNG ⚔️')
    print("=" * 65)
    print('🔵 ĐỘI XANH:', ', '.join(blue_team))
    print('🔴 ĐỘI ĐỎ:', ', '.join(red_team))

    typ = '🔵 ĐỘI XANH CÓ LỢI THẾ 🔵' if prob_blue >= 0.5 else '🔴 ĐỘI ĐỎ CÓ LỢI THẾ 🔴'
    
    print('🏆 ĐÁNH GIÁ:', typ)
    print('   •TỈ LỆ THẮNG ĐỘI XANH:', str(round(prob_blue * 100, 2)) + '%')
    print('   •TỈ LỆ THẮNG ĐỘI ĐỎ:', str(round(prob_red * 100, 2)) + '%')
    print("=" * 65 + "\n")

blue_team=["Jayce", "Lee Sin", "Ahri", "Kai'Sa", "Nautilus"]
red_team =["K'Sante", "Viego", "Orianna", "Jinx", "Thresh"]
pred(blue_team, red_team)


⚔️ KẾT QUẢ DỰ ĐOÁN TỈ LỆ THẮNG ⚔️
🔵 ĐỘI XANH: Jayce, Lee Sin, Ahri, Kai'Sa, Nautilus
🔴 ĐỘI ĐỎ: K'Sante, Viego, Orianna, Jinx, Thresh
🏆 ĐÁNH GIÁ: 🔴 ĐỘI ĐỎ CÓ LỢI THẾ 🔴
   •TỈ LỆ THẮNG ĐỘI XANH: 48.12%
   •TỈ LỆ THẮNG ĐỘI ĐỎ: 51.88%

